In [1]:
import sys

sys.path.append("../training_data")

In [2]:
from utils.utils import *

# structures_path = "1.Minimal_structures/Minimal_structures"

# Cif.path = structures_path
# Cif.original_cifs_path = "/home/fnerin/Desktop/allodb_new/src/data" # <pdb>_updated.cif.gz

# Functions

## Get representative's sites annos

Dictionary with the site's residues, modulators, chains, and its uniprots. For each uniprot, the corresponding site residues, chains and its residues, and range.

In [3]:
def get_sites_ups(p, sites):
    sites_ups = {}
    for sid, site in sites.items():
        # The annotation is based on Uniprots, so equivalent sites of the site are assumed to have the same Uniprots
        site_res = site["site"]

        # Skip site if any of the chains of the site do not have more than 3 Uniprot-associated residues participating in it
        if any(
            len(ares.query("pdbx_sifts_xref_db_acc != '?'")) < 3 
            for aid, ares in site_res.groupby("label_asym_id")
        ):
            continue

        # For each Uniprot in the site, annotate the chain IDs, the residues of the site and of the chains (that have Uniprots) and the min-max of the Uniprot
        site_ups = {}
        for u, ures in site_res.query("pdbx_sifts_xref_db_acc != '?'").groupby("pdbx_sifts_xref_db_acc"):
            asym_ids = ures.label_asym_id.unique().tolist()
            chains_res = p.residues.query(f"label_asym_id in {asym_ids} and pdbx_sifts_xref_db_acc == '{u}'") # maybe simplify
            site_ups[u] = {
                "ulabel_asym_id": asym_ids, 
                "usiteres": ures,
                "ures": chains_res,
                "umin": chains_res.pdbx_sifts_xref_db_num.astype(int).min(),
                "umax": chains_res.pdbx_sifts_xref_db_num.astype(int).max()
            }
        if len(site_ups) == 0: continue

        sites_ups[sid] = {
            "siteres": site_res,
            "label_asym_id": site_res.query("pdbx_sifts_xref_db_acc != '?'").label_asym_id.unique().tolist(),
            "uniprots": site_ups,
            "modulator": site['mod'].label_asym_id.unique().tolist()
        }
        
    return sites_ups

## Detect ligands close to site

Any elements closer than 6A from the center of geometry of a passed site to detect true apos.

In [4]:
import pymol2

def get_apo_ligands(newpdb, newp, site, threshold=6):
     # Define the PyMOL-style selection of the modulator residues
    sele = " or ".join(
        f"{res['label_asym_id']}/{res['auth_asym_id']}/{res['auth_comp_id']}`{res['auth_seq_id']}{res['pdbx_PDB_ins_code'].replace('?', '')}/*"
        for i, res in site.iterrows()
    )

    with pymol2.PyMOL() as pymol, newp.cif.ciff() as f:
        pymol.cmd.feedback(
            "disable", "executive", "details"
        )  # to silence "ExecutiveLoad-Detail: Detected mmCIF"
        pymol.cmd.load(f.name)

        # Crete a pseudoatom in the center of mass of the site's residues
        pymol.cmd.pseudoatom("pa", selection=sele)

        # Retrieve all atoms within the threshold of the selection
        atoms = pymol.cmd.get_model(f"(br. all within {threshold} of pa) and not (pa or {sele})")
        
    # Process the atom selection to obtain residue identifiers
    residue_ids = set(
        tuple(
            (
                a.segi, a.chain, a.resn,
                a.resi_number, a.ins_code or '?' # pdbx_PDB_ins_code or "?" if none
            ) 
            for a in atoms.atom
        )
    )

    # Transform the PyMOL-derived residue identifiers into a standard table of residues that can be used to retrieve the rows/residues from the parent structure's .residues table    
    residues = (
        newp.residues.merge(
            pd.DataFrame(
                residue_ids,
                columns=[
                    "label_asym_id", "auth_asym_id", "auth_comp_id",
                    "auth_seq_id", "pdbx_PDB_ins_code"
                ],
                dtype=str
            )
        )
        .query("auth_comp_id != 'HOH'")
        .query(f"label_asym_id not in {site.label_asym_id.unique().tolist()}")
    )

    # If there are any residues
    if len(residues) > 0:
        # Get the cif field with entity data        
        edata = pd.DataFrame(newpdb.cif.data["_entity"], dtype=str)
        
        # Get dictionaries of the info of only non-polymer items
        ligands = [
            e
            for e in (
                edata
                .query(f"id in {residues.label_entity_id.unique().tolist()}")[
                    ["pdbx_description", "type"]
                ].to_dict("records")
            )
        ]
        
        ions = all(e["pdbx_description"].lower().endswith(" ion") for e in ligands)

        # Return the info
        if ions: # if found molecules is all ions: no lig
            return {
            ("apo", "has_lig"): False,
            ("apo", "lig_types"): ["ion",],
            ("apo", "lig"): residues
        }
        else: #
            return {
                ("apo", "has_lig"): True,
                ("apo", "lig_types"): list(set(e["type"] for e in ligands)),
                ("apo", "lig"): residues
            }
    else:
        return {("apo", "has_lig"): False}

## Check overlaps in a new putative apo pdb

In [5]:
from utils.new_pdbs import Pdb
import numpy as np
import gzip

In [6]:
def check_pdb(newpdb, pdb, sites_ups, ups):
    newpdb = Pdb(newpdb.lower())

    # If the new PDB doesn't have a column for Uniprotts or doesn't have all Uniprots of holo, skip
    if "pdbx_sifts_xref_db_acc" not in newpdb.residues.columns: return
    if not all(u in newpdb.residues.pdbx_sifts_xref_db_acc.unique() for u in ups): return

    # Use structure of the assembly if it's different from the model and it isn't repetition-based
    newp = newpdb
    try:
        if newpdb.assembly is not None and newpdb.assembly._repetitions is None:
            newp = newpdb.assembly
    except: # fix/patch for structures with multiple models to use only first
        with newpdb._extended_temp_ciff({
            "_atom_site": newpdb.atoms.query("pdbx_PDB_model_num == '1'").to_dict(orient="list")
        }) as f:
            newpdb.cif._cif_content = gzip.compress(f.read().encode())
        if newpdb.assembly is not None and newpdb.assembly._repetitions is None:
            newp = newpdb.assembly
    # we assume that all sites_uniprots are going to be found in the pdb, AND:
    # we don't care that the chains with the ups of interest have other ups (chimeras)

    # Check each Uniprot-based site for overlaps
    for sid, site in sites_ups.items():
        # Obtain the residues of the Uniprot-based site in the new structure, and skip if not found
        merged_site = (
            newp.residues
            .query(f"pdbx_PDB_model_num == '1'")
            .merge(
                site["siteres"],
                on=['pdbx_sifts_xref_db_acc', 'pdbx_sifts_xref_db_num'],
                suffixes = (None, '_del')
            )
            .loc[:, lambda x: [c for c in x if not c.endswith("_del")]]
        )
        if len(merged_site) == 0:
            continue

        # Get the ligands found in the correspnding Uniprot-based site, if any
        ligands = {
            asym_id: get_apo_ligands(newpdb, newp, merged_siteres)
            for asym_id, merged_siteres in merged_site.groupby("label_asym_id")
        }

        # Create the base Uniprot-site dictionary with the holo information
        sd = {
            ("allodb", "pdb"): pdb.entry_id,
            ("allodb", "site_id"): sid,
            ("allodb", "label_asym_id"): site["label_asym_id"],
            ("allodb", "mod"): site["modulator"],
            ("allodb", "siteres"): site["siteres"],
        }
        # Create the base Uniprot-site dictionary with the apo information at the whole-site level
        apo_sd = {
            ("apo", "site"): round(
                (
                    len( merged_site[['pdbx_sifts_xref_db_acc', 'pdbx_sifts_xref_db_num']].drop_duplicates() ) 
                    / len( site["siteres"][['pdbx_sifts_xref_db_acc', 'pdbx_sifts_xref_db_num']].drop_duplicates() )
                ),               
                2
            ),
            ("apo", "siteres"): merged_site,
            ("apo", "any_apo_chain"): any(not c[("apo", "has_lig")] for c in ligands.values()),
            ("apo", "ligands"): ligands,
        }
        # Yield a dictinary for each Uniprot on the original Uniprot-site, containing sd and apo_sd together with other Uniprot-specific information
        for u, ud in site["uniprots"].items():
            d = sd.copy()

            # Obtain the Uniprot residues of the structure and calculate chain and site overlaps
            ures = newp.residues.query(f"pdbx_sifts_xref_db_acc == '{u}' and pdbx_PDB_model_num == '1'")
            if len(ures) == 0:
                break
            umin, umax = ures.pdbx_sifts_xref_db_num.astype(int).min(), ures.pdbx_sifts_xref_db_num.astype(int).max()
            usite = ures.merge(
                ud["usiteres"], 
                on=['pdbx_sifts_xref_db_acc', 'pdbx_sifts_xref_db_num'],
                suffixes = (None, '_del')
            ).loc[:, lambda x: [c for c in x if not c.endswith("_del")]]
                            
            udrange = range(ud["umin"], ud["umax"]+1)
            urange = range(umin, umax+1)
            merge = ud["ures"].drop_duplicates().merge(ures[['pdbx_sifts_xref_db_acc', 'pdbx_sifts_xref_db_num']].drop_duplicates())

            # Return the Uniprot-specific dictionary with all holo, apo, Uniprot and chain and site overlap info
            d.update({
                ("allodb", "uniprot"): u,
                **{("allodb", k): v for k, v in ud.items() if k != "ures"},
                ("apo", "pdb"): newpdb.entry_id, 
                ("apo", "label_asym_id"): ures.label_asym_id.unique().tolist(), 
                **apo_sd,
                ("apo", "umin"): umin,
                ("apo", "umax"): umax,
                ("apo", "usite"): round(
                    len(
                        usite[
                            ['pdbx_sifts_xref_db_acc', 'pdbx_sifts_xref_db_num']
                        ].drop_duplicates()
                    )  / len( ud["usiteres"][['pdbx_sifts_xref_db_acc', 'pdbx_sifts_xref_db_num']].drop_duplicates() ),
                    2
                ),
                ("overlap", "a_in_h_minmax"): len(np.intersect1d(udrange, urange)) / len(urange),
                ("overlap", "h_in_a_minmax"): len(np.intersect1d(udrange, urange)) / len(udrange),
                ("overlap", "a_in_h_merge"): len(merge) / len( ures[['pdbx_sifts_xref_db_acc', 'pdbx_sifts_xref_db_num']].drop_duplicates() ),
                ("overlap", "h_in_a_merge"): len(merge) / len( ud["ures"][['pdbx_sifts_xref_db_acc', 'pdbx_sifts_xref_db_num']].drop_duplicates() ),
            })
            yield d

## Function

In [7]:
from rcsbsearchapi import rcsb_attributes
from functools import partial

# Apos of extra set

## Data

In [8]:
from tqdm import tqdm

In [9]:
import os, sys, pickle
from tqdm.notebook import tqdm

In [10]:
extra_holos_featuresf = "../training_data/7.Extra_set/features.pkl"

with open(extra_holos_featuresf, "rb") as f:
    extra_holos_featuresd = pickle.load(f)

extra_holos_featuresd

{'7gqu':     Residues                                                          \
          pdb label_entity_id label_asym_id label_seq_id auth_asym_id   
 0       7gqu               1             A           12            A   
 1       7gqu               1             A           13            A   
 2       7gqu               1             A           14            A   
 3       7gqu               1             A           15            A   
 4       7gqu               1             A           16            A   
 ..       ...             ...           ...          ...          ...   
 414     7gqu               1             A          426            A   
 415     7gqu               1             A          427            A   
 416     7gqu               1             A          428            A   
 417     7gqu               1             A          429            A   
 418     7gqu               1             A          430            A   
 
                                   Label 

In [11]:
extra_holos_sitesf = "../training_data/7.Extra_set/news_sites.pkl"

with open(extra_holos_sitesf, "rb") as f:
    extra_holos_sites = {k: v for k, v in pickle.load(f).items() if k in extra_holos_featuresd}

extra_holos_sites

{'7gqu': [{'mod':      label_comp_id label_asym_id label_entity_id label_seq_id  \
   3420           X1L             D               4            .   
   
        pdbx_PDB_ins_code auth_seq_id auth_comp_id auth_asym_id  \
   3420                 ?        1002          X1L            A   
   
        pdbx_PDB_model_num pdbx_label_index pdbx_sifts_xref_db_name  \
   3420                  1             1002                       ?   
   
        pdbx_sifts_xref_db_acc pdbx_sifts_xref_db_num pdbx_sifts_xref_db_res  
   3420                      ?                      ?                      ?  ,
   'site':    label_comp_id label_asym_id label_entity_id label_seq_id pdbx_PDB_ins_code  \
   0            VAL             A               1           55                 ?   
   1            MET             A               1           56                 ?   
   2            ALA             A               1           57                 ?   
   3            THR             A               1         

In [12]:
len(extra_holos_featuresd), len(extra_holos_sites)

(9, 9)

##### 8uk6

Already know the only other structure is the AlphaFold model.

In [13]:
extra_holos_featuresd.pop("8uk6")
extra_holos_sites.pop("8uk6")

[{'mod':      label_comp_id label_asym_id label_entity_id label_seq_id  \
  4678           WVK             C               3            .   
  
       pdbx_PDB_ins_code auth_seq_id auth_comp_id auth_asym_id  \
  4678                 ?         802          WVK            A   
  
       pdbx_PDB_model_num pdbx_label_index pdbx_sifts_xref_db_name  \
  4678                  1              802                       ?   
  
       pdbx_sifts_xref_db_acc pdbx_sifts_xref_db_num pdbx_sifts_xref_db_res  
  4678                      ?                      ?                      ?  ,
  'site':    label_comp_id label_asym_id label_entity_id label_seq_id pdbx_PDB_ins_code  \
  0            GLY             A               1          281                 ?   
  1            PHE             A               1          282                 ?   
  2            LEU             A               1          283                 ?   
  3            HIS             A               1          284                 ?  

## Apos

In [14]:
extra_apos_path = "pdb_ensemble"

os.makedirs(extra_apos_path, exist_ok=True)

In [15]:
extra_apos_original_cifs_path = f"{extra_apos_path}/origcifs"
os.makedirs(extra_apos_original_cifs_path, exist_ok=True)

extra_apos_structures_path = f"{extra_apos_path}/cifs"
os.makedirs(extra_apos_structures_path, exist_ok=True)

# extra_apos_pockets_path = f"{extra_apos_path}/pockets"
# os.makedirs(extra_apos_pockets_path, exist_ok=True)

# extra_apos_features_path = extra_apos_path
# os.makedirs(f"{extra_apos_features_path}/features", exist_ok=True)

### Run

In [16]:
Cif.path = "../training_data/7.Extra_set/cifs"
Cif.original_cifs_path = "../training_data/7.Extra_set/origcifs" # <pdb>_updated.cif.gz
Pdb.path = extra_apos_structures_path

In [58]:
def get_extra_apos(pdb):
    pdb = Cif(pdb)
    if "pdbx_sifts_xref_db_acc" not in pdb.residues.columns:
        print(f"{pdb.entry_id}: No Uniprot column in original pdb")
        return []
    
    # protein or others. allo modulators. Ions?
    sites_ups = get_sites_ups(pdb, {i: s for i, s in enumerate(extra_holos_sites[pdb.entry_id])})
    if len(sites_ups) == 0:
        print(f"{pdb.entry_id}: No valid Uniprots in any site")

    return sites_ups
    
    # # Including fix for 8jp0 "P32418-2"
    # ups = list(set(u.replace("-2", "") for sd in sites_ups.values() for u in sd["uniprots"].keys()))
    
    
    # pdbs = False
    # for q in (
    #     rcsb_attributes.rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession == up
    #     for up in ups
    # ):
    #     pdbs = q if not pdbs else pdbs & q
    
    # if not pdbs:
    #     print(f"{pdb.entry_id}: No valid Uniprots in any site")
    #     return []
        
    # pdbs = set(i.lower() for i in pdbs()) - set(extra_holos_featuresd.keys())
    # if len(pdbs) == 0: 
    #     print(f"{pdb.entry_id}: No other PDBs of Uniprots")
    #     return []

    # for newpdb in tqdm(pdbs, smoothing=0, desc=pdb.entry_id):
    #     for r in check_pdb(newpdb, pdb, sites_ups, ups):
    #         yield r

In [61]:
test = get_extra_apos("8jp0")

In [62]:
test

{0: {'siteres':    label_comp_id label_asym_id label_entity_id label_seq_id pdbx_PDB_ins_code  \
  0            GLU             A               1          132                 ?   
  1            THR             A               1          133                 ?   
  2            VAL             A               1          134                 ?   
  3            SER             A               1          135                 ?   
  4            LEU             A               1          137                 ?   
  5            THR             A               1          138                 ?   
  6            ALA             A               1          141                 ?   
  7            HIS             A               1          200                 ?   
  8            VAL             A               1          203                 ?   
  9            VAL             A               1          206                 ?   
  10           THR             A               1          207            

In [64]:
test[0]["siteres"]

,label_comp_id,label_asym_id,label_entity_id,label_seq_id,pdbx_PDB_ins_code,auth_seq_id,auth_comp_id,auth_asym_id,pdbx_PDB_model_num,pdbx_label_index,pdbx_sifts_xref_db_name,pdbx_sifts_xref_db_acc,pdbx_sifts_xref_db_num,pdbx_sifts_xref_db_res
0,GLU,A,1,132,?,132,GLU,A,1,132,UNP,P32418-2,132,E
1,THR,A,1,133,?,133,THR,A,1,133,UNP,P32418-2,133,T
2,VAL,A,1,134,?,134,VAL,A,1,134,UNP,P32418-2,134,V
3,SER,A,1,135,?,135,SER,A,1,135,UNP,P32418-2,135,S
4,LEU,A,1,137,?,137,LEU,A,1,137,UNP,P32418-2,137,L
5,THR,A,1,138,?,138,THR,A,1,138,UNP,P32418-2,138,T
6,ALA,A,1,141,?,141,ALA,A,1,141,UNP,P32418-2,141,A
7,HIS,A,1,200,?,200,HIS,A,1,200,UNP,P32418-2,200,H
8,VAL,A,1,203,?,203,VAL,A,1,203,UNP,P32418-2,203,V
9,VAL,A,1,206,?,206,VAL,A,1,206,UNP,P32418-2,206,V


In [67]:
test[0]["uniprots"].keys()

dict_keys(['P32418-2'])

In [72]:
test[0]["uniprots"]["P32418-2"]["usiteres"].replace({"pdbx_sifts_xref_db_acc": {"P32418-2": "P32418"}})

,label_comp_id,label_asym_id,label_entity_id,label_seq_id,pdbx_PDB_ins_code,auth_seq_id,auth_comp_id,auth_asym_id,pdbx_PDB_model_num,pdbx_label_index,pdbx_sifts_xref_db_name,pdbx_sifts_xref_db_acc,pdbx_sifts_xref_db_num,pdbx_sifts_xref_db_res
0,GLU,A,1,132,?,132,GLU,A,1,132,UNP,P32418,132,E
1,THR,A,1,133,?,133,THR,A,1,133,UNP,P32418,133,T
2,VAL,A,1,134,?,134,VAL,A,1,134,UNP,P32418,134,V
3,SER,A,1,135,?,135,SER,A,1,135,UNP,P32418,135,S
4,LEU,A,1,137,?,137,LEU,A,1,137,UNP,P32418,137,L
5,THR,A,1,138,?,138,THR,A,1,138,UNP,P32418,138,T
6,ALA,A,1,141,?,141,ALA,A,1,141,UNP,P32418,141,A
7,HIS,A,1,200,?,200,HIS,A,1,200,UNP,P32418,200,H
8,VAL,A,1,203,?,203,VAL,A,1,203,UNP,P32418,203,V
9,VAL,A,1,206,?,206,VAL,A,1,206,UNP,P32418,206,V


In [43]:
extra_matchesf = f"{extra_apos_path}/matches.pkl"

In [44]:
extra_df = pd.DataFrame(columns=pd.MultiIndex.from_tuples(
    (('allodb', 'pdb'), ('allodb', 'site_id'), ('allodb', 'label_asym_id'), ('allodb', 'mod'), ('allodb', 'uniprot'), ('allodb', 'ulabel_asym_id'), ('allodb', 'usiteres'), ('allodb', 'umin'), ('allodb', 'umax'), ('apo', 'pdb'), ('apo', 'label_asym_id'), ('apo', 'site'), ('apo', 'siteres'), ('apo', 'ligands'), ('apo', 'any_apo_chain'), ('apo', 'umin'), ('apo', 'umax'), ('apo', 'usite'), ('overlap', 'a_in_h_minmax'), ('overlap', 'h_in_a_minmax'), ('overlap', 'a_in_h_merge'), ('overlap', 'h_in_a_merge'))
))

In [45]:
# Retrieve processed elements from the DataFrame
extra_processed = extra_df[("allodb", "pdb")].unique().tolist()

In [21]:
# Load the existing DataFrame if it exists
if os.path.exists(extra_matchesf):
    extra_df = pd.read_pickle(extra_matchesf)
    extra_processed.extend(extra_df[("allodb", "pdb")].unique().tolist())

In [46]:
# Process the reps keys in chunks
for p in ["8jp0",]:
    if p not in extra_processed:
        p_df = pd.DataFrame(get_extra_apos(p))
        if len(p_df):
            p_df.columns = pd.MultiIndex.from_tuples(p_df.columns)
            extra_df = pd.concat([extra_df, p_df])
            extra_df.to_pickle(extra_matchesf)
        extra_processed.append(p)

{0: {'siteres':    label_comp_id label_asym_id label_entity_id label_seq_id pdbx_PDB_ins_code  \
0            GLU             A               1          132                 ?   
1            THR             A               1          133                 ?   
2            VAL             A               1          134                 ?   
3            SER             A               1          135                 ?   
4            LEU             A               1          137                 ?   
5            THR             A               1          138                 ?   
6            ALA             A               1          141                 ?   
7            HIS             A               1          200                 ?   
8            VAL             A               1          203                 ?   
9            VAL             A               1          206                 ?   
10           THR             A               1          207                 ?   
11          

8jp0:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
sites_ups[0]

In [29]:
extra_df

Empty DataFrame
Columns: [(allodb, pdb), (allodb, site_id), (allodb, label_asym_id), (allodb, mod), (allodb, uniprot), (allodb, ulabel_asym_id), (allodb, usiteres), (allodb, umin), (allodb, umax), (apo, pdb), (apo, label_asym_id), (apo, site), (apo, siteres), (apo, ligands), (apo, any_apo_chain), (apo, umin), (apo, umax), (apo, usite), (overlap, a_in_h_minmax), (overlap, h_in_a_minmax), (overlap, a_in_h_merge), (overlap, h_in_a_merge)]
Index: []

[0 rows x 22 columns]

In [30]:
# Process the reps keys in chunks
for p in tqdm(extra_holos_featuresd, smoothing=0):
    if p not in extra_processed:
        p_df = pd.DataFrame(get_extra_apos(p))
        if len(p_df):
            p_df.columns = pd.MultiIndex.from_tuples(p_df.columns)
            extra_df = pd.concat([extra_df, p_df])
            extra_df.to_pickle(extra_matchesf)
        extra_processed.append(p)

  0%|          | 0/8 [00:00<?, ?it/s]

7gqu:   0%|          | 0/21 [00:00<?, ?it/s]

/tmp/ipykernel_172997/2230244043.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  extra_df = pd.concat([extra_df, p_df])


7yg5:   0%|          | 0/4 [00:00<?, ?it/s]

8aq6:   0%|          | 0/20 [00:00<?, ?it/s]

8f4s:   0%|          | 0/3022 [00:00<?, ?it/s]

8jp0:   0%|          | 0/4 [00:00<?, ?it/s]

8qni:   0%|          | 0/23 [00:00<?, ?it/s]

8v81:   0%|          | 0/57 [00:00<?, ?it/s]

9dnm:   0%|          | 0/15 [00:00<?, ?it/s]

## Analysis

In [21]:
extra_df = extra_df.reset_index(drop=True)
extra_df

allodb                                                    \
       pdb site_id label_asym_id  mod uniprot ulabel_asym_id   
0     7gqu       0           [A]  [D]  Q14191            [A]   
1     7gqu       0           [A]  [D]  Q14191            [A]   
2     7gqu       0           [A]  [D]  Q14191            [A]   
3     7gqu       0           [A]  [D]  Q14191            [A]   
4     7gqu       0           [A]  [D]  Q14191            [A]   
..     ...     ...           ...  ...     ...            ...   
141   9dnm       0           [C]  [D]  P29066            [C]   
142   9dnm       0           [C]  [D]  P29066            [C]   
143   9dnm       0           [C]  [D]  P29066            [C]   
144   9dnm       0           [C]  [D]  P29066            [C]   
145   9dnm       0           [C]  [D]  P29066            [C]   

                                                                   apo  ...  \
                                              usiteres umin umax   pdb  ...   
0       label_comp_id label_asym_id label_entity_id...  527  945  7gqt  ...   
1       label_comp_id label_asym_id label_entity_id...  527  945  9og3  ...   
2       label_comp_id label_asym_id label_entity_id...  527  945  8yle  ...   
3       label_comp_id label_asym_id label_entity_id...  527  945  9og8  ...   
4       label_comp_id label_asym_id label_entity_id...  527  945  6yhr  ...   
..                                                 ...  ...  ...   ...  ...   
141    label_comp_id label_asym_id label_entity_id ...    6  391  8i0n  ...   
142    label_comp_id label_asym_id label_entity_id ...    6  391  9bt8  ...   
143    label_comp_id label_asym_id label_entity_id ...    6  391  8go8  ...   
144    label_comp_id label_asym_id label_entity_id ...    6  391  9cx3  ...   
145    label_comp_id label_asym_id label_entity_id ...    6  391  8hsv  ...   

                                                                           \
                                               ligands any_apo_chain umin   
0                   {'A': {('apo', 'has_lig'): False}}          True  527   
1    {'A': {('apo', 'has_lig'): True, ('apo', 'lig_...         False  526   
2                   {'A': {('apo', 'has_lig'): False}}          True  526   
3    {'A': {('apo', 'has_lig'): True, ('apo', 'lig_...         False  526   
4                   {'A': {('apo', 'has_lig'): False}}          True  528   
..                                                 ...           ...  ...   
141  {'A': {('apo', 'has_lig'): True, ('apo', 'lig_...         False    7   
142                 {'E': {('apo', 'has_lig'): False}}          True    6   
143  {'A': {('apo', 'has_lig'): True, ('apo', 'lig_...         False    7   
144                 {'E': {('apo', 'has_lig'): False}}          True    6   
145  {'A': {('apo', 'has_lig'): False}, 'C': {('apo...          True    4   

                      overlap                                          \
     umax usite a_in_h_minmax h_in_a_minmax a_in_h_merge h_in_a_merge   
0     945   1.0      1.000000      1.000000     1.000000     0.988067   
1     946   1.0      0.995249      1.000000     0.995157     0.980907   
2     945   1.0      0.997619      1.000000     0.997596     0.990453   
3     946   1.0      0.995249      1.000000     0.995227     0.995227   
4    1072   1.0      0.766972      0.997613     0.772643     0.997613   
..    ...   ...           ...           ...          ...          ...   
141   368   1.0      1.000000      0.937824     0.886740     0.975684   
142   356   1.0      1.000000      0.909326     0.944785     0.936170   
143   368   1.0      1.000000      0.937824     0.886740     0.975684   
144   356   1.0      1.000000      0.909326     0.953560     0.936170   
145   394   1.0      0.987212      1.000000     0.906336     1.000000   

                                                allodb  
                                               siteres  
0       label_comp_id label_asym_id label_entity_id...  
1       label_comp_i

In [22]:
len(extra_df[("allodb", "pdb")].unique())

7

##### No site overlap

In [23]:
set(
    extra_df[("allodb", "pdb")].unique()
) - set(
    extra_df[extra_df[("apo", "site")] >= 0.8][("allodb", "pdb")].unique()
) 

set()

In [24]:
extra_df[("allodb", "pdb")].value_counts(), \
extra_df[extra_df[("apo", "site")] >= 0.8][("allodb", "pdb")].value_counts()

((allodb, pdb)
 8f4s    71
 8v81    21
 8qni    16
 9dnm    15
 8aq6    11
 7gqu     9
 7yg5     3
 Name: count, dtype: int64,
 (allodb, pdb)
 8f4s    71
 8v81    21
 9dnm    15
 8aq6    11
 8qni    11
 7gqu     9
 7yg5     3
 Name: count, dtype: int64)

##### 8qni

In [28]:
(
    extra_df[extra_df[("allodb", "pdb")] == "8qni"]
    .drop(columns=[("allodb", "usiteres"), ("allodb", "siteres"), ("apo", "siteres"), ("apo", "ligands"),])
    .sort_values(("apo", "site"))
)

allodb                                                               apo  \
       pdb site_id label_asym_id  mod uniprot ulabel_asym_id umin umax   pdb   
108   8qni       0           [A]  [B]  Q13191            [A]   38  420  2ldr   
95    8qni       0           [A]  [B]  Q13191            [A]   38  420  8vw4   
98    8qni       0           [A]  [B]  Q13191            [A]   38  420  3vgo   
97    8qni       0           [A]  [B]  Q13191            [A]   38  420  3pfv   
103   8qni       0           [A]  [B]  Q13191            [A]   38  420  8vw5   
99    8qni       0           [A]  [B]  Q13191            [A]   38  420  8gcy   
96    8qni       0           [A]  [B]  Q13191            [A]   38  420  8qtj   
94    8qni       0           [A]  [B]  Q13191            [A]   38  420  9fqi   
101   8qni       0           [A]  [B]  Q13191            [A]   38  420  9fqj   
100   8qni       0           [A]  [B]  Q13191            [A]   38  420  3zni   
104   8qni       0           [A]  [B]  Q13191            [A]   38  420  9fqh   
102   8qni       0           [A]  [B]  Q13191            [A]   38  420  8qtg   
105   8qni       0           [A]  [B]  Q13191            [A]   38  420  8qtk   
106   8qni       0           [A]  [B]  Q13191            [A]   38  420  8qng   
107   8qni       0           [A]  [B]  Q13191            [A]   38  420  8qth   
109   8qni       0           [A]  [B]  Q13191            [A]   38  420  8qnh   

                                                            overlap  \
    label_asym_id  site any_apo_chain umin umax usite a_in_h_minmax   
108           [A]  0.23          True  351  426  0.23      0.921053   
95            [A]  0.77         False   38  341  0.77      1.000000   
98            [A]  0.77          True   43  339  0.77      1.000000   
97            [A]  0.77          True   44  344  0.77      1.000000   
103           [A]  0.77          True   38  341  0.77      1.000000   
99            [A]  1.00         False   38  427  1.00      0.982051   
96            [A]  1.00         False   37  427  1.00      0.979540   
94            [A]  1.00         False   37  427  1.00      0.979540   
101           [A]  1.00         False   37  427  1.00      0.979540   
100           [A]  1.00         False   38  427  1.00      0.982051   
104           [A]  1.00         False   37  427  1.00      0.979540   
102           [A]  1.00         False   37  427  1.00      0.979540   
105           [A]  1.00         False   37  427  1.00      0.979540   
106           [A]  1.00         False   38  427  1.00      0.982051   
107           [A]  1.00         False   37  427  1.00      0.979540   
109           [A]  1.00         False   37  427  1.00      0.979540   

                                             
    h_in_a_minmax a_in_h_merge h_in_a_merge  
108      0.182768     0.881579     0.177719  
95       0.793734     1.000000     0.806366  
98       0.775457     1.000000     0.761273  
97       0.785901     1.000000     0.798408  
103      0.793734     1.000000     0.806366  
99       1.000000     0.981333     0.976127  
96       1.000000     0.979167     0.997347  
94       1.000000     0.979167     0.997347  
101      1.000000     0.979167     0.997347  
100      1.000000     0.966667     1.000000  
104      1.000000     0.979167     0.997347  
102      1.000000     0.979167     0.997347  
105      1.000000     0.979167     0.997347  
106      1.000000     0.981771     1.000000  
107      1.000000     0.976684     1.000000  
109      1.000000     0.979167     0.997347

0.77 site presence in the apo is also acceptable. **Some of the PDBs have h_in_a overlap of less than 0.8, and will be taken into account in the next step**

<br>

In [29]:
extra_df[("allodb", "pdb")].value_counts(), \
extra_df[extra_df[("apo", "site")] >= 0.77][("allodb", "pdb")].value_counts()

((allodb, pdb)
 8f4s    71
 8v81    21
 8qni    16
 9dnm    15
 8aq6    11
 7gqu     9
 7yg5     3
 Name: count, dtype: int64,
 (allodb, pdb)
 8f4s    71
 8v81    21
 8qni    15
 9dnm    15
 8aq6    11
 7gqu     9
 7yg5     3
 Name: count, dtype: int64)

In [30]:
extra_filt_df = extra_df[extra_df[("apo", "site")] >= 0.77]

##### No chain overlap

In [33]:
# No chain overlap
## Checked with the % of the holo residues inside the apo because we can use bigger structures, and "cut" the extra part
set(
    extra_filt_df[("allodb", "pdb")].unique()
) - set(
    extra_filt_df[extra_filt_df[("overlap", "h_in_a_merge")] >= 0.76][("allodb", "pdb")].unique()
) 

set()

In [34]:
extra_filt_df[("allodb", "pdb")].value_counts(), \
extra_filt_df[extra_filt_df[("overlap", "h_in_a_merge")] >= 0.76][("allodb", "pdb")].value_counts()

((allodb, pdb)
 8f4s    71
 8v81    21
 8qni    15
 9dnm    15
 8aq6    11
 7gqu     9
 7yg5     3
 Name: count, dtype: int64,
 (allodb, pdb)
 8f4s    71
 8v81    21
 8qni    15
 9dnm    15
 8aq6    11
 7gqu     9
 7yg5     2
 Name: count, dtype: int64)

##### 7yg5

In [35]:
(
    extra_filt_df[extra_filt_df[("allodb", "pdb")] == "7yg5"]
    .drop(columns=[("allodb", "usiteres"), ("allodb", "siteres"), ("apo", "siteres"), ("apo", "ligands"),])
)

allodb                                                                apo  \
      pdb site_id label_asym_id  mod uniprot ulabel_asym_id umin  umax   pdb   
9    7yg5       0           [A]  [V]  Q15878            [A]   76  1871  8epm   
10   7yg5       0           [A]  [V]  Q15878            [A]   76  1871  7xlq   
11   7yg5       0           [A]  [V]  Q15878            [A]   76  1871  8epl   

                                                            overlap  \
   label_asym_id  site any_apo_chain umin  umax usite a_in_h_minmax   
9            [A]  0.86          True   89  1730  0.86           1.0   
10           [A]  1.00          True   76  1871  1.00           1.0   
11           [A]  1.00          True   85  1851  1.00           1.0   

                                            
   h_in_a_minmax a_in_h_merge h_in_a_merge  
9       0.914254     0.996973     0.749052  
10      1.000000     1.000000     1.000000  
11      0.983853     0.986656     0.952995

The 8epm will also be a valid apo.

<br>

In [36]:
extra_filt_df = extra_filt_df[extra_filt_df[("overlap", "h_in_a_merge")] >= 0.74]

##### No apos

In [37]:
set(
    extra_filt_df[("allodb", "pdb")].unique()
) - set(
    extra_filt_df[extra_filt_df[("apo", "any_apo_chain")] == True][("allodb", "pdb")].unique()
) 

set()

In [38]:
extra_filt_df[("allodb", "pdb")].value_counts(), \
extra_filt_df[extra_filt_df[("apo", "any_apo_chain")] == True][("allodb", "pdb")].value_counts()

((allodb, pdb)
 8f4s    71
 8v81    21
 8qni    15
 9dnm    15
 8aq6    11
 7gqu     9
 7yg5     3
 Name: count, dtype: int64,
 (allodb, pdb)
 8f4s    57
 8v81    19
 9dnm    11
 8aq6     9
 7gqu     5
 7yg5     3
 8qni     3
 Name: count, dtype: int64)

In [39]:
(
    extra_filt_df[ extra_filt_df[("apo", "any_apo_chain")] == False ]
    .drop(columns=[("allodb", "usiteres"), ("allodb", "siteres"), ("apo", "siteres"),])
    .sort_values([("allodb", "pdb")], ascending=False)
    [[("allodb", "pdb"), ("apo", "pdb"), ("apo", "ligands")]]
    .to_dict(orient="records")
)

[{('allodb', 'pdb'): '9dnm',
  ('apo', 'pdb'): '8go8',
  ('apo',
   'ligands'): {'A': {('apo', 'has_lig'): True,
    ('apo', 'lig_types'): ['polymer'],
    ('apo',
     'lig'):   label_comp_id label_asym_id label_entity_id label_seq_id pdbx_PDB_ins_code  \
    1           LEU             C               1          335                 ?   
    2           ASP             C               1          337                 ?   
    3           LEU             C               1          338                 ?   
    
      auth_seq_id auth_comp_id auth_asym_id pdbx_PDB_model_num pdbx_label_index  \
    1         335          LEU            B                  1              335   
    2         337          ASP            B                  1              337   
    3         338          LEU            B                  1              338   
    
      pdbx_sifts_xref_db_name pdbx_sifts_xref_db_acc pdbx_sifts_xref_db_num  \
    1                     UNP                 P29066                  

- The ones from 9dnm are all to be considered apo, they just have a crystal contact.
- From 8v81: not mod-free
- From 8qni, 3zni and 8vw4 only have co-crystallization molecules
- From 8f4s, 8ov3, 8bsd, 9eml, 9gue, 9gny, 8ov4, 9guy, 8ov1, 8bzv, 9q8h, 9grp, 8oto, 8ov2 have a co-crystallization molecule and has an adjacent ligand ¿orthosteric site? (not 8f4y)
- From 8aq6, 8aqi has mod, 5ibo has a fatty acid
- From 7gqu, all are mods

In [40]:
no_apo = extra_filt_df[ extra_filt_df[("apo", "any_apo_chain")] == False ]

In [45]:
extra_filt_df = pd.concat((
    extra_filt_df[extra_filt_df[("apo", "any_apo_chain")] == True],
    no_apo[(no_apo[("allodb", "pdb")] == "9dnm")],
    no_apo[(no_apo[("allodb", "pdb")] == "8qni") & (no_apo[("apo", "pdb")].isin(["3zni", "8vw4"]))],
    no_apo[(no_apo[("allodb", "pdb")] == "8f4s") & (no_apo[("apo", "pdb")] != "8f4y")]
))

##### Multiple sites

In [49]:
{
    p: {
        sid: sid in extra_df[("allodb", "site_id")].unique()
        for sid in sids
    }
    for p, g in extra_df.groupby(("allodb", "pdb"))
    for sids in [
        extra_df.loc[lambda x: x[("allodb", "pdb")] == p]
        .loc[:, ("allodb", "site_id")]
        .unique(),
    ]
    if len(sids) > 1
}

{}

None.

<br>

##### Heteromers

In [50]:
[
    p for p, g in extra_df.groupby(("allodb", "pdb"))
    if len(
        extra_df.loc[lambda x: x[("allodb", "pdb")] == p]
        .loc[:, ("allodb", "uniprot")]
        .unique()
    ) > 1
]

[]

None

<br>

## Cifs and chains

In [18]:
Cif.path = extra_apos_structures_path
Cif.original_cifs_path = extra_apos_original_cifs_path # <pdb>_updated.cif.gz
Pdb.path = extra_apos_structures_path

In [19]:
aposf = f"{extra_apos_path}/apos.pkl"

In [79]:
apos = {}

for holo, rows in  tqdm(extra_filt_df.groupby(("allodb", "pdb"), sort=False)):
    apos[holo] = {}
    print(holo)
    for apo, g in tqdm(rows.groupby(("apo", "pdb"), sort=False), total=len(rows)):
        assert len(g) == 1, "What"
        pdb = Pdb(apo)
        
        origciff = f"{extra_apos_original_cifs_path}/{apo}_updated.cif.gz"
        if not os.path.isfile(origciff):
            with open(origciff, "wb") as f:
                f.write(pdb.cif._cif_content)

        # ASSUMING ALL ARE MONOMERS
        apo_chains = tuple(sorted(
            [
                c
                for c, cd in g[("apo", "ligands")].item().items() 
                if not cd[("apo", "has_lig")]
            ] 
            or g[("apo", "label_asym_id")].item()
        ))
        apos[holo][apo] = apo_chains[0]


with open(aposf, "wb") as f:
    pickle.dump(apos, f)
apos

  0%|          | 0/7 [00:00<?, ?it/s]

7gqu


  0%|          | 0/5 [00:00<?, ?it/s]

7yg5


  0%|          | 0/3 [00:00<?, ?it/s]

...

8v81


  0%|          | 0/19 [00:00<?, ?it/s]

9dnm


  0%|          | 0/15 [00:00<?, ?it/s]

{'7gqu': {'7gqt': 'A', '8yle': 'A', '6yhr': 'A', '8pfp': 'A', '7gqs': 'A'},
 '7yg5': {'8epm': 'A', '7xlq': 'A', '8epl': 'A'},
 '8aq6': {'7snx': 'A',
  '7snr': 'A',
  '7sny': 'A',
  '7snw': 'A',
  '7sns': 'A',
  '7vsx': 'A',
  '8aqh': 'A',
  '5b0u': 'A',
  '7snt': 'A'},
 '8f4s': {'6wq3': 'A',
  '7jz0': 'A',
  '7lw4': 'A',
  '8tyj': 'A',
  '9gtf': 'A',
  '6wvn': 'A',
  '9eun': 'A',
  '7bq7': 'A',
  '8ot0': 'A',
  '9gs4': 'A',
  '8rze': 'A',
  '7lw3': 'A',
  '7jhe': 'A',
  '9emv': 'A',
  '8rva': 'A',
  '7r1t': 'A',
  '7koa': 'A',
  '6wrz': 'A',
  '7r1u': 'A',
  '6yz1': 'A',
  '7ult': 'A',
  '8rv8': 'A',
  '6wjt': 'A',
  '8s8x': 'A',
  '6xkm': 'A',
  '6w61': 'A',
  '8rv9': 'A',
  '6wkq': 'A',
  '9p6p': 'A',
  '8otr': 'A',
  '8rzc': 'A',
  '7c2j': 'A',
  '8s8w': 'A',
  '8rv4': 'A',
  '8c5m': 'A',
  '9guf': 'A',
  '7c2i': 'A',
  '8osx': 'A',
  '8rv7': 'A',
  '7jpe': 'A',
  '7l6r': 'A',
  '7jyy': 'A',
  '6w4h': 'A',
  '9grq': 'A',
  '9emj': 'A',
  '8vuo': 'A',
  '7l6t': 'A',
  '7jib': 'A',
  

In [80]:
apos

{'7gqu': {'7gqt': 'A', '8yle': 'A', '6yhr': 'A', '8pfp': 'A', '7gqs': 'A'},
 '7yg5': {'8epm': 'A', '7xlq': 'A', '8epl': 'A'},
 '8aq6': {'7snx': 'A',
  '7snr': 'A',
  '7sny': 'A',
  '7snw': 'A',
  '7sns': 'A',
  '7vsx': 'A',
  '8aqh': 'A',
  '5b0u': 'A',
  '7snt': 'A'},
 '8f4s': {'6wq3': 'A',
  '7jz0': 'A',
  '7lw4': 'A',
  '8tyj': 'A',
  '9gtf': 'A',
  '6wvn': 'A',
  '9eun': 'A',
  '7bq7': 'A',
  '8ot0': 'A',
  '9gs4': 'A',
  '8rze': 'A',
  '7lw3': 'A',
  '7jhe': 'A',
  '9emv': 'A',
  '8rva': 'A',
  '7r1t': 'A',
  '7koa': 'A',
  '6wrz': 'A',
  '7r1u': 'A',
  '6yz1': 'A',
  '7ult': 'A',
  '8rv8': 'A',
  '6wjt': 'A',
  '8s8x': 'A',
  '6xkm': 'A',
  '6w61': 'A',
  '8rv9': 'A',
  '6wkq': 'A',
  '9p6p': 'A',
  '8otr': 'A',
  '8rzc': 'A',
  '7c2j': 'A',
  '8s8w': 'A',
  '8rv4': 'A',
  '8c5m': 'A',
  '9guf': 'A',
  '7c2i': 'A',
  '8osx': 'A',
  '8rv7': 'A',
  '7jpe': 'A',
  '7l6r': 'A',
  '7jyy': 'A',
  '6w4h': 'A',
  '9grq': 'A',
  '9emj': 'A',
  '8vuo': 'A',
  '7l6t': 'A',
  '7jib': 'A',
  

# Predictions

In [20]:
import os, sys, subprocess
import pandas as pd
from tqdm.notebook import tqdm

In [21]:
sys.path.append("..")

In [22]:
from predict import \
get_cif, get_clean_pdb, Cif, Site, get_pockets, Pocket, \
get_pdb_features, get_pockets_features, \
prepare_data, model, view_pockets
# get_features

/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/autogluon/common/utils/utils.py:78: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
INFO:.prody:ProDy is configured: verbosity='none'
/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [23]:
os.makedirs(f"{extra_apos_path}/predictions", exist_ok=True)

In [24]:
holos_ups = {
    '7gqu': 'Q14191',
    '7yg5': 'Q15878',
    '8aq6': 'Q9GV45',
    '8f4s': 'P0DTD1',
    '8qni': 'Q13191',
    # '8uk6': 'A0A1D8PQM9',
    '8v81': 'P13569',
    '9dnm': 'P29066',
    '8jp0': 'P32418'
} # already with -2 correction

In [25]:
overlapsf = f"{extra_apos_path}/predictions.pkl"

In [28]:
if os.path.isfile(overlapsf):
    overlaps = pd.read_pickle(overlapsf)
else:
    overlaps = {}

## Results

- For **7gqu** all 4 structures have an acceptable pocket. 8pfp's pocket includes the allo site but is very big, and 7gqt/7gqs are small pockets within the allo site. They all have a bound ligand in a different site and AlloPockets is successful at discriminating.
- For **7yg5** the results are still incorrect and moreover they are not consistent.
- For **8aq6**, the results are mostly negative as with the originally-chosen apo, predicting the orthosteric site. For 7vsx and 7sny the predicted pocket extends a bit through the surface.
- For 8f4s (so far), predictions are consistent with the originally-chosen apo; but the pocket is the one that contains the ortho ligand and extends a bit (~35% of residues) to the allosteric site.
- For **8v81**, pockets are wrongly predicted and the only two with high overlap are enormous pockets spanning the whole structure.
- For 9dnm, results aren't great either and the ones showing overlap are again big pockets and mostly do not cover the surface-shallow site, but come from behind or from the side. The pockets would also need to be compared with the non-annotated pocket.
- For **8qni** the results are good, with 3/4 apos (+ original apo) being correct predictions.

In [26]:
apos = pd.read_pickle(aposf)
dict(sorted({k: len(v) for k, v in apos.items()}.items(), key=lambda x: x[1], reverse=True))

{'8f4s': 70,
 '8v81': 19,
 '9dnm': 15,
 '8aq6': 9,
 '7gqu': 5,
 '8qni': 5,
 '7yg5': 3}

In [27]:
overlaps = pd.read_pickle(overlapsf)
dict(sorted({k: len(v) for k, v in overlaps.items()}.items(), key=lambda x: x[1], reverse=True))

{'8f4s': 68,
 '8v81': 17,
 '9dnm': 14,
 '8aq6': 7,
 '7gqu': 4,
 '8qni': 4,
 '7yg5': 2}

**8v81**
- 6o1v: top4 18%
- 7svd: -
- 7sv7: top3 8%
- 8eig: top3 27%
- 6msm: top5 24%
- 9dw9: top5 8%
- 9dw5: top6 27%
- 8v7z: top3 16%
- 8eio: top5 7%
- 8gls: top4 25%
- 7svr: top2 23%
- 9dw8: top2 54%
- 8eiq: top4 40%
- 6o2p: top5 19%
- 9dw7: top3 63%
- 9dw4, 8ej1

**7yg5**
- 8epm:
    - top1: could coincide with modulator in 8F0Q (chimera Na channel) when taking account symmetry
    - top2: -
    - top3: -
    - top4: pore
    - top5: -
- 8epl:
    - top1: -
    - top2: -, similar to top3 from 8epm
    - top3: -, similar to top2 from 8epm
    - top4: coincides with modulator in 8QUD

In [150]:
holo = "7gqu"
pd.DataFrame(overlaps[holo]).T.sort_values("overlap")

,pocket,overlap,site,merge
7gqt,pocket5,0.068966,label_comp_id label_asym_id label_entity_id...,label_comp_id label_asym_id label_entity_id ...
7gqs,pocket4,0.068966,label_comp_id label_asym_id label_entity_id...,label_comp_id label_asym_id label_entity_id ...
8yle,pocket6,0.448276,label_comp_id label_asym_id label_entity_id...,label_comp_id label_asym_id label_entity_id...
8pfp,pocket1,0.586207,label_comp_id label_asym_id label_entity_id...,label_comp_id label_asym_id label_entity_id...


In [155]:
pdb = "8pfp"
toppocket = 1

preds = pd.read_pickle(f"pdb_ensemble/predictions/{pdb}/preds.pkl").iloc[:5]; print(preds)
cif = Cif(pdb, f'pdb_ensemble/predictions/{pdb}/{pdb}.cif')
view_pockets(
    cif,
    pockets={preds.iloc[toppocket-1].name: {"color": "green"}},
    site_residues=cif.residues.merge(
        extra_holos_sites[holo][0]["site"][["pdbx_sifts_xref_db_acc", "pdbx_sifts_xref_db_num"]]
    ),
    path=f'pdb_ensemble/predictions/{pdb}',
)

          Allosteric score
pocket1           0.610110
pocket3           0.185555
pocket2           0.025168
pocket4           0.002601
pocket10          0.001652


PDBeMolstar(bg_color='#F7F7F7', color_data={'data': [{'struct_asym_id': 'A', 'representation': 'cartoon', 'rep…

In [66]:
apos = pd.read_pickle(f"{extra_apos_path}/apos.pkl")

In [73]:
dict(sorted({k: len(v) for k, v in apos.items()}.items(), key=lambda x: x[1], reverse=True))

{'8f4s': 70,
 '8v81': 19,
 '9dnm': 15,
 '8aq6': 9,
 '7gqu': 5,
 '8qni': 5,
 '7yg5': 3}

In [98]:
apos["8v81"]

{'6o1v': 'A',
 '9dw8': 'A',
 '7svr': 'A',
 '8gls': 'A',
 '8eio': 'A',
 '8v7z': 'A',
 '8eiq': 'A',
 '5uak': 'A',
 '8fzq': 'A',
 '9dw7': 'B',
 '9dw5': 'A',
 '6msm': 'A',
 '9dw4': 'B',
 '8eig': 'A',
 '7sv7': 'A',
 '7svd': 'A',
 '8ej1': 'A',
 '9dw9': 'A',
 '6o2p': 'A'}